# NB-InvSym 1: Broken-inversion-symmetry descriptors (regression)

Builds a family of descriptors that quantify *how much* inversion (and mirror) symmetry is broken, as
ma'am asked: "put an inversion/mirror through the atom, see if it maps back into itself, and if not
quantify the leftover in distance and angle, weighted by heavy Z." Unlike GRDF (which is
inversion-invariant and so blind to symmetry breaking), these are built to be **zero for a
centrosymmetric site and grow with distortion**.

**All parameters were fixed from the geometry scan in `nb_invsym_00_parameter_analysis.ipynb`:**
- **Adaptive cutoff** `Rc_struct = min(Rc_target, 0.95 * wrap_radius)` with `Rc_target = 3.5`.
  Block 2b showed a *fixed* 4.5 A reaches into a periodic image for 100% of structures and even 4.0 A
  for ~99%; the safe wrap radius (half the shortest in-plane repeat) medians only ~2.2-2.9 A. So the
  cutoff must scale per structure or the inversion test matches atoms against their own periodic ghosts.
- **Pairing on ANY species** (Block 3: 0% of atoms lacked a same-species neighbour in range, but
  forcing same-species pulls in spurious far matches; ANY gives the physical 'is there an atom where
  the image should be').
- **Matched-fraction form, not mean residual** (Blocks 3-4: nearest-image distance is bimodal -
  a 'matched' lobe near 0 and an 'unmatched' lobe past 2 A - so a mean sits in the dead zone and is
  uninformative; the informative quantity is the *fraction of neighbour-pairs that find a partner
  within tolerance*). Tolerance is **scanned** (0.2/0.3/0.5/0.75) and chosen by alpha-correlation, not
  hard-set.
- **Global-centre search: coarse 3x3 grid** (Block 5: the minimum residual was identical at 3x3, 5x5,
  9x9 - no need to refine).
- **Z-weighting central, EN-weighting demoted** (Block 6: EN range does not track the target;
  Z-weighting encodes SOC ~ Z^4 which is a *magnitude* statement, exactly what regression needs).
- **Buckling is a 99-only feature** (Block 6: identically 0 for all dresselhaus and unmatched, so on
  the 124 it would predict via a class-constant, not physics; flagged and audited there, used freely
  on the 99).

**Regression only. No classifier.** Reported on three slices, 99 and 124 being the priority:
- **99**  = rashba only (transferable headline target)
- **124** = rashba + dresselhaus (all splitting compounds)
- **200** = + unmatched as alpha=0 (zero-inflated; reference only - the 77 zeros make R2 easy)

Place in `Keshav-DDP/Broken-inversion-symmetry/`.

## Cell 1: Config + imports (paths identical to nb_expand_dataset)

In [ ]:
import os, glob, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")
from pymatgen.core.structure import Structure
from pymatgen.core.periodic_table import Element

BASE_DIR = os.path.abspath("..")
SET_DIRS = {
    "rashba":      os.path.join(BASE_DIR, "Inverse-design", "rashba"),
    "dresselhaus": os.path.join(BASE_DIR, "Inverse-design", "dresselhaus"),
    "unmatched":   os.path.join(BASE_DIR, "Inverse-design", "unmatched"),
}
# precomputed E_pfrac CSVs (uid, Formula, alpha_R, E_pfrac_VBM, E_pfrac_CBM) - same as nb_expand
DESC_DIR = os.path.join(BASE_DIR, "Weight-contribution", "contribution-model")
DESC_CSVS = {
    "rashba":      os.path.join(DESC_DIR, "desc_E_p1.0_rashba.csv"),
    "dresselhaus": os.path.join(DESC_DIR, "desc_E_p1.0_dresselhaus.csv"),
    "unmatched":   os.path.join(DESC_DIR, "desc_E_p1.0_unmatched.csv"),
}
RESULTS_DIR = os.path.join(".", "invsym-results")
os.makedirs(RESULTS_DIR, exist_ok=True)

print("PATH CHECK")
for k, p in {**{f"dir:{s}": d for s, d in SET_DIRS.items()},
             **{f"csv:{s}": c for s, c in DESC_CSVS.items()}}.items():
    print(f"  [{'OK' if os.path.exists(p) else 'MISSING'}] {k:18s} {p}")

## Cell 2: Master table (one row per uid) + load structures

Same construction as nb_expand: alpha = max over a uid's appearances, unmatched forced to 0. We also
keep `set` and `is_split` so we can slice 99 / 124 / 200. E_pfrac and radius_mean are the proven
transferable baseline features (the 0.66-era core) so we can measure what the inversion descriptors
*add* on top, not in isolation.

In [ ]:
parts = []
for s, csv in DESC_CSVS.items():
    c = pd.read_csv(csv)
    need = ["uid", "Formula", "alpha_R", "E_pfrac_VBM", "E_pfrac_CBM"]
    miss = [x for x in need if x not in c.columns]
    assert not miss, f"{s}: missing {miss}; has {list(c.columns)}"
    c = c[need].copy(); c["set"] = s
    if s == "unmatched":
        c["alpha_R"] = 0.0
    parts.append(c)
raw = pd.concat(parts, ignore_index=True)

g = raw.groupby("uid")
master = pd.DataFrame({
    "Formula":     g["Formula"].first(),
    "alpha":       g["alpha_R"].max(),
    "alpha_mean":  g["alpha_R"].mean(),
    "alpha_min":   g["alpha_R"].min(),
    "E_pfrac_VBM": g["E_pfrac_VBM"].first(),
    "E_pfrac_CBM": g["E_pfrac_CBM"].first(),
    "set":         g["set"].apply(lambda v: "rashba" if "rashba" in set(v)
                                  else ("dresselhaus" if "dresselhaus" in set(v) else "unmatched")),
    "is_split":    g["set"].apply(lambda v: int(any(x in ("rashba", "dresselhaus") for x in v))),
}).reset_index()

def find_folder(uid, vdir):
    for folder in glob.glob(os.path.join(vdir, "*")):
        name = os.path.basename(folder); idx = name.rfind("-")
        if idx != -1 and name[idx+1:] == uid:
            return folder
    return None

def load_std(folder):
    p = os.path.join(folder, "POSCAR_std")
    return Structure.from_file(p) if os.path.exists(p) else None

structures = {}; missing = []
for _, row in master.iterrows():
    uid = row["uid"]; vdir = SET_DIRS[row["set"]]
    folder = find_folder(uid, vdir)
    s = load_std(folder) if folder else None
    if s is None or max(site.specie.Z for site in s) <= 3:
        missing.append((row["set"], uid)); continue
    structures[uid] = s
master = master[master["uid"].isin(structures)].reset_index(drop=True)

print("compounds with structure:", len(master))
print("by set:", master["set"].value_counts().to_dict())
print("missing:", len(missing), missing[:5])
print(master.head(3).to_string(index=False))

## Cell 3: Geometry primitives - adaptive Rc + neighbour vectors

`wrap_radius` = half the shortest in-plane periodic repeat (the distance at which a neighbour would be
a periodic image rather than a distinct atom). `adaptive_rc` caps the cutoff below that. `neighbor_vectors`
returns, per atom, the displacement vectors to neighbours within the *structure's own* Rc, plus their Z
and species - the raw material for every inversion/mirror residual.

In [ ]:
RC_TARGET = 3.5     # from Block 2b: largest target that stays near the safe wrap radius after capping
RC_SAFETY = 0.95    # stay just inside the periodic image

def wrap_radius(s):
    abc = np.array(s.lattice.abc); vac = int(np.argmax(abc))
    inplane = [ax for ax in range(3) if ax != vac]
    M = s.lattice.matrix; a1, a2 = M[inplane[0]], M[inplane[1]]
    min_repeat = min(np.linalg.norm(a1), np.linalg.norm(a2),
                     np.linalg.norm(a1 + a2), np.linalg.norm(a1 - a2))
    return 0.5 * min_repeat, vac

def adaptive_rc(s):
    wr, vac = wrap_radius(s)
    return min(RC_TARGET, RC_SAFETY * 2 * wr), vac   # 0.95 * min_repeat where min_repeat = 2*wr

def neighbor_vectors(s, rc):
    alln = s.get_all_neighbors(rc); out = []
    for i, site in enumerate(s):
        ci = site.coords; v, zj, sym = [], [], []
        for nb in alln[i]:
            if nb.nn_distance <= 1e-3:
                continue
            v.append(np.array(nb.coords) - ci); zj.append(nb.specie.Z); sym.append(nb.specie.symbol)
        out.append(dict(zi=site.specie.Z, sym_i=site.specie.symbol,
                        v=np.array(v) if v else np.zeros((0, 3)),
                        zj=np.array(zj), sym=np.array(sym, dtype=object)))
    return out

# quick look at the adaptive cutoff actually used, per set
rc_used = {k: [] for k in ["rashba", "dresselhaus", "unmatched"]}
for uid, s in structures.items():
    rc, _ = adaptive_rc(s); rc_used[master.set_index("uid").loc[uid, "set"]].append(rc)
print("adaptive Rc actually used (A), per set:  mean [min, max]")
for k, v in rc_used.items():
    if v: print(f"  {k:12s} {np.mean(v):.2f}  [{min(v):.2f}, {max(v):.2f}]")

## Cell 4: Descriptor family A - INVERSION residual (local atom centre)

Ma'am's core operation. For each atom (treated as the local inversion centre) and each neighbour
vector **v**, the inversion image sits at **-v**. Find the nearest actual neighbour to that image point
(ANY species) and record three things:
  - **matched** (1 if that nearest atom is within tolerance `tol`, else 0) -> the locally-centrosymmetric
    fraction. This is the headline form (mean residual is degenerate per Blocks 3-4).
  - **distance residual** |r_partner - r_v| (radial leftover, ma'am's 'distance' deviation),
  - **angle residual** the angle between the actual partner and the ideal image direction (ma'am's 'theta').

Each is produced **plain**, **sqrt(Zi*Zj)-weighted**, and **Z^4-weighted** (heavy-atom emphasis;
SOC ~ Z^4). Per structure we aggregate over atoms with both **mean and max**. Tolerance is scanned.

In [ ]:
TOLS = [0.2, 0.3, 0.5, 0.75]   # scanned; alpha-correlation in Cell 9 picks the best

def _aggregate(per_atom_vals, weights_per_atom, prefix, feat):
    """mean and max over atoms, plain and Z-weighted, into feat dict."""
    v = np.array(per_atom_vals, dtype=float)
    if v.size == 0:
        for agg in ("mean", "max"):
            feat[f"{prefix}_{agg}"] = 0.0
        return
    feat[f"{prefix}_mean"] = float(v.mean())
    feat[f"{prefix}_max"]  = float(v.max())

def inversion_local(nd, tol):
    """Per-atom matched-fraction, distance residual, angle residual (plain + Z^4 weighted).
    Returns per-atom arrays so the caller can aggregate."""
    frac, frac_z4, dres, dres_z4, ares, ares_z4 = [], [], [], [], [], []
    for a in nd:
        V = a["v"]
        if len(V) < 2:
            continue
        norms = np.linalg.norm(V, axis=1)
        Zi = a["zi"]; Zj = a["zj"]
        m, d, ang, w4 = [], [], [], []
        for j in range(len(V)):
            target = -V[j]
            dd = np.linalg.norm(V - target, axis=1)
            b = int(np.argmin(dd))
            m.append(1.0 if dd[b] <= tol else 0.0)
            d.append(abs(norms[b] - norms[j]))
            ct = np.clip(np.dot(V[b], target) / (norms[b] * norms[j] + 1e-12), -1, 1)
            ang.append(np.arccos(ct))
            w4.append(max(Zi, Zj[j]) ** 4)
        m = np.array(m); d = np.array(d); ang = np.array(ang); w4 = np.array(w4)
        # per-atom: the matched fraction is 'how centrosymmetric is this site'; broken = 1 - frac
        frac.append(1.0 - m.mean())                       # broken fraction (higher = more broken)
        frac_z4.append(1.0 - (m * w4).sum() / w4.sum())   # heavy-atom-weighted broken fraction
        dres.append(d.mean()); dres_z4.append((d * w4).sum() / w4.sum())
        ares.append(ang.mean()); ares_z4.append((ang * w4).sum() / w4.sum())
    return dict(frac=frac, frac_z4=frac_z4, dres=dres, dres_z4=dres_z4, ares=ares, ares_z4=ares_z4)

def build_inversion_local(s):
    rc, _ = adaptive_rc(s); nd = neighbor_vectors(s, rc)
    feat = {}
    for tol in TOLS:
        r = inversion_local(nd, tol)
        t = f"{tol:.2f}".replace(".", "")
        _aggregate(r["frac"],    None, f"inv_broken_t{t}",      feat)
        _aggregate(r["frac_z4"], None, f"inv_broken_z4_t{t}",   feat)
    # distance/angle residuals do not depend on tol -> compute once (use first tol's nd result)
    r0 = inversion_local(nd, TOLS[0])
    _aggregate(r0["dres"],    None, "inv_dres",    feat)
    _aggregate(r0["dres_z4"], None, "inv_dres_z4", feat)
    _aggregate(r0["ares"],    None, "inv_ares",    feat)
    _aggregate(r0["ares_z4"], None, "inv_ares_z4", feat)
    return feat

print("family A (inversion, local centre) ready")

## Cell 5: Descriptor family B - MIRROR residual (local atom centre)

Block 4 hint: dresselhaus (non-polar) had the *highest* inversion residual - because their broken
operation is a mirror, not inversion. Ma'am explicitly said 'put a mirror plane in here.' So we repeat
family A with the operation being reflection through the plane normal to the out-of-plane (vacuum)
axis: the image of **v** = (vx, vy, vz) is (vx, vy, -vz). For a structure with a real horizontal mirror
this is zero; out-of-plane (Janus-type) asymmetry makes it grow. Same plain / Z^4, mean / max.

In [ ]:
def mirror_local(nd, vac, tol):
    """Reflection through the in-plane mirror (flip the vacuum-axis component)."""
    frac, frac_z4, dres, dres_z4, ares, ares_z4 = [], [], [], [], [], []
    flip = np.ones(3); flip[vac] = -1.0
    for a in nd:
        V = a["v"]
        if len(V) < 2:
            continue
        norms = np.linalg.norm(V, axis=1)
        Zi = a["zi"]; Zj = a["zj"]
        m, d, ang, w4 = [], [], [], []
        for j in range(len(V)):
            target = V[j] * flip                      # mirror image
            dd = np.linalg.norm(V - target, axis=1)
            b = int(np.argmin(dd))
            m.append(1.0 if dd[b] <= tol else 0.0)
            d.append(abs(norms[b] - norms[j]))
            ct = np.clip(np.dot(V[b], target) / (norms[b] * norms[j] + 1e-12), -1, 1)
            ang.append(np.arccos(ct))
            w4.append(max(Zi, Zj[j]) ** 4)
        m = np.array(m); d = np.array(d); ang = np.array(ang); w4 = np.array(w4)
        frac.append(1.0 - m.mean()); frac_z4.append(1.0 - (m * w4).sum() / w4.sum())
        dres.append(d.mean()); dres_z4.append((d * w4).sum() / w4.sum())
        ares.append(ang.mean()); ares_z4.append((ang * w4).sum() / w4.sum())
    return dict(frac=frac, frac_z4=frac_z4, dres=dres, dres_z4=dres_z4, ares=ares, ares_z4=ares_z4)

def build_mirror_local(s):
    rc, vac = adaptive_rc(s); nd = neighbor_vectors(s, rc)
    feat = {}
    for tol in TOLS:
        r = mirror_local(nd, vac, tol); t = f"{tol:.2f}".replace(".", "")
        _aggregate(r["frac"],    None, f"mir_broken_t{t}",    feat)
        _aggregate(r["frac_z4"], None, f"mir_broken_z4_t{t}", feat)
    r0 = mirror_local(nd, vac, TOLS[0])
    _aggregate(r0["dres"],    None, "mir_dres",    feat)
    _aggregate(r0["dres_z4"], None, "mir_dres_z4", feat)
    _aggregate(r0["ares"],    None, "mir_ares",    feat)
    _aggregate(r0["ares_z4"], None, "mir_ares_z4", feat)
    return feat

print("family B (mirror, local centre) ready")

## Cell 6: Descriptor family C - GLOBAL best-fit inversion centre

Instead of every atom being its own centre, search for the single centre that best inverts the *whole*
cell, and report the leftover. A perfectly centrosymmetric crystal drives this to zero; a Janus/buckled
one leaves a finite floor - a rigorous continuous symmetry measure. Block 5 showed a coarse **3x3**
grid over fractional centres in [0,0.5] already finds the minimum (no change at 5x5/9x9), so this is
cheap. We report the minimised mean residual, plus a Z^4-weighted version (heavy atoms dominate the
sum). Single scalar per structure (no per-atom mean/max needed).

In [ ]:
GLOBAL_GRID = 3   # 3x3 from Block 5

def build_global_center(s):
    cart = s.cart_coords; Z = np.array([site.specie.Z for site in s], dtype=float)
    L = s.lattice
    best = np.inf; best_z4 = np.inf
    for fa in np.linspace(0, 0.5, GLOBAL_GRID):
        for fb in np.linspace(0, 0.5, GLOBAL_GRID):
            c = L.get_cartesian_coords([fa, fb, 0.5])
            res = np.array([np.linalg.norm(cart - (2 * c - cart[i]), axis=1).min()
                            for i in range(len(s))])
            mean_res = res.mean()
            w4 = Z ** 4
            mean_res_z4 = (res * w4).sum() / w4.sum()
            if mean_res < best:
                best = mean_res
            if mean_res_z4 < best_z4:
                best_z4 = mean_res_z4
    return dict(glob_inv_resid=float(best), glob_inv_resid_z4=float(best_z4))

print("family C (global best-fit inversion centre) ready")

## Cell 7: Descriptor family D - structural companions (buckling, top/bottom asymmetry, wrap scale)

Cheap structure-only quantities flagged in the scan. **Buckling** (heavy-vs-light sublattice
z-separation) is included but tagged so we can exclude it from the 124/200 slices (Block 6: 0 for all
dresselhaus/unmatched -> on those slices it is a class constant, not physics). **Top/bottom
coordination asymmetry**: per atom, compare the summed neighbour vector above vs below its own plane -
the out-of-plane asymmetry that produces the Rashba field. **wrap_radius / min_repeat** are real length
scales and harmless as features.

In [ ]:
def build_structural(s):
    feat = {}
    abc = np.array(s.lattice.abc); vac = int(np.argmax(abc))
    zc = s.cart_coords[:, vac]
    syms = np.array([site.specie.symbol for site in s]); Zs = np.array([site.specie.Z for site in s])
    # buckling: heavy-vs-light sublattice z separation (TAGGED: 99-only safe)
    heavy = syms == syms[np.argmax(Zs)]; light = syms == syms[np.argmin(Zs)]
    feat["buckling"] = float(abs(zc[heavy].mean() - zc[light].mean())) if (heavy.any() and light.any()) else 0.0
    # top/bottom coordination asymmetry: per atom, net out-of-plane component of neighbour vectors
    rc, _ = adaptive_rc(s); nd = neighbor_vectors(s, rc)
    asym = []
    for a in nd:
        V = a["v"]
        if len(V) == 0:
            continue
        asym.append(abs(V[:, vac].sum()))   # |net z of neighbour bonds| -> 0 if top/bottom balanced
    asym = np.array(asym)
    feat["topbot_asym_mean"] = float(asym.mean()) if asym.size else 0.0
    feat["topbot_asym_max"]  = float(asym.max())  if asym.size else 0.0
    # length scales
    wr, _ = wrap_radius(s)
    feat["wrap_radius"] = float(wr)
    return feat

print("family D (structural companions) ready")

## Cell 8: Build all descriptors for every structure -> feature table

Runs families A-D over all structures and merges into the master table. `BUCKLING_99_ONLY` records the
feature(s) that must be dropped from the 124/200 slices.

In [ ]:
recs = []
for n, (uid, s) in enumerate(structures.items()):
    feat = {"uid": uid}
    feat.update(build_inversion_local(s))
    feat.update(build_mirror_local(s))
    feat.update(build_global_center(s))
    feat.update(build_structural(s))
    recs.append(feat)
    if (n + 1) % 40 == 0:
        print(f"  {n+1}/{len(structures)}")
df_feat = pd.DataFrame(recs)

work = master.merge(df_feat, on="uid")
INVSYM_FEATS = [c for c in df_feat.columns if c != "uid"]
BUCKLING_99_ONLY = ["buckling"]   # exclude from 124/200 (class-constant there)

print(f"\nwork table: {len(work)} compounds, {len(INVSYM_FEATS)} inversion-symmetry features")
print("families:")
for tag, pref in [("A inversion", "inv_"), ("B mirror", "mir_"),
                  ("C global", "glob_"), ("D structural", ("buckling", "topbot", "wrap"))]:
    fs = [c for c in INVSYM_FEATS if (c.startswith(pref) if isinstance(pref, str) else c.startswith(pref))]
    print(f"  {tag:14s} {len(fs):3d}")
work.to_csv(os.path.join(RESULTS_DIR, "invsym_features.csv"), index=False)
print("saved invsym_features.csv")

## Cell 9: alpha-correlation audit per slice (which features track magnitude, and best tolerance)

Regression-only relevance test: Spearman correlation of each feature with alpha, computed separately on
the **99** and **124** slices (the priority targets). Spearman because the relationships may be monotonic
but not linear. This (a) tells us which inversion-symmetry features are worth feeding the model, and
(b) picks the tolerance whose `inv_broken` / `mir_broken` correlates best - replacing the hard-coded 0.3
from the parameter scan. Buckling is shown but its 124 correlation is flagged as suspect (class constant).

In [ ]:
from scipy.stats import spearmanr

def slice_frame(name):
    if name == "99":  return work[work["set"] == "rashba"]
    if name == "124": return work[work["is_split"] == 1]
    return work

rows = []
for f in INVSYM_FEATS:
    r = {"feature": f}
    for name in ["99", "124", "200"]:
        fr = slice_frame(name); x = fr[f].values; y = fr["alpha"].values
        r[f"rho_{name}"] = spearmanr(x, y).correlation if np.std(x) > 0 else 0.0
    rows.append(r)
corr = pd.DataFrame(rows)
corr["abs99"] = corr["rho_99"].abs()
corr = corr.sort_values("abs99", ascending=False)

pd.set_option("display.float_format", lambda v: f"{v:.3f}")
print("top 20 by |Spearman rho| on the 99 slice (rho_99, rho_124, rho_200):")
print(corr[["feature", "rho_99", "rho_124", "rho_200"]].head(20).to_string(index=False))

# choose best tolerance for the matched-fraction descriptors, by |rho_99|
print("\nbest tolerance for inv_broken / mir_broken (by |rho_99| of the _mean variant):")
for fam in ["inv_broken", "mir_broken"]:
    cands = corr[corr["feature"].str.startswith(fam) & corr["feature"].str.endswith("_mean")
                 & ~corr["feature"].str.contains("z4")]
    if len(cands):
        bestrow = cands.iloc[0]
        print(f"  {fam:12s} -> {bestrow['feature']:24s} rho_99={bestrow['rho_99']:.3f}")
corr.to_csv(os.path.join(RESULTS_DIR, "invsym_alpha_correlation.csv"), index=False)

## Cell 10: Modeling helpers (XGB + forward selection, LOO) - same protocol as nb_expand

Identical XGBRegressor config and LOO discipline as the rest of the project so numbers are comparable.
`reg_report` prints 5-fold and LOO R2. `reg_forward` does greedy forward selection (best-1, add the
feature that most improves 5-fold R2, repeat) - the small-sample-safe protocol you asked for, since
with 99-124 samples fitting all features at once overfits.

In [ ]:
from sklearn.model_selection import LeaveOneOut, KFold, cross_val_predict
from sklearn.metrics import r2_score, mean_absolute_error
from xgboost import XGBRegressor

XGB_REG_PARAMS = dict(n_estimators=100, max_depth=3, learning_rate=0.1, subsample=0.8,
                      colsample_bytree=0.8, reg_alpha=1.0, reg_lambda=1.0, random_state=42, verbosity=0)
_kf = KFold(n_splits=5, shuffle=True, random_state=42)

def _cv(frame, feats, y, cv):
    X = frame[feats].fillna(0).values
    return cross_val_predict(XGBRegressor(**XGB_REG_PARAMS), X, y, cv=cv)

def reg_report(frame, feats, y, label):
    yl = _cv(frame, feats, y, LeaveOneOut())
    y5 = _cv(frame, feats, y, _kf)
    print(f"  {label:34s} 5f R2={r2_score(y,y5):.3f}  LOO R2={r2_score(y,yl):.3f}  MAE={mean_absolute_error(y,yl):.3f}")
    return r2_score(y, yl)

def reg_forward(frame, y, pool, seed, upto):
    chosen = list(seed)
    while len(chosen) < upto and len(chosen) < len(pool) + len(seed):
        rem = [f for f in pool if f not in chosen]
        if not rem:
            break
        cand = sorted(((r2_score(y, _cv(frame, chosen + [f], y, _kf)), f) for f in rem), reverse=True)
        chosen.append(cand[0][1])
    return chosen

print("modeling helpers ready")

## Cell 11: Per-family forward selection on each slice (does each family add over the baseline?)

The baseline is the proven transferable core (E_pfrac_VBM, E_pfrac_CBM) - roughly the 0.66-era model
without k-path features. For each slice (99, 124, 200) and each descriptor family, we seed with the
baseline and forward-select up to +5 family features, reporting LOO R2 at each step. This isolates what
*each* family adds. Buckling is dropped from the family-D pool on the 124/200 slices.

In [ ]:
BASELINE = ["E_pfrac_VBM", "E_pfrac_CBM"]
FAMILIES = {
    "A inversion":  [c for c in INVSYM_FEATS if c.startswith("inv_")],
    "B mirror":     [c for c in INVSYM_FEATS if c.startswith("mir_")],
    "C global":     [c for c in INVSYM_FEATS if c.startswith("glob_")],
    "D structural": [c for c in INVSYM_FEATS if c.startswith(("buckling", "topbot", "wrap"))],
}

for sl in ["99", "124", "200"]:
    fr = slice_frame(sl).reset_index(drop=True); y = fr["alpha"].values
    print(f"\n===== SLICE {sl}  (n={len(fr)}) =====")
    base_r2 = reg_report(fr, BASELINE, y, "baseline (E_pfrac VBM+CBM)")
    for fam, pool in FAMILIES.items():
        p = [f for f in pool if not (sl != "99" and f in BUCKLING_99_ONLY)]
        if not p:
            continue
        chosen = reg_forward(fr, y, p, BASELINE, upto=len(BASELINE) + 5)
        added = chosen[len(BASELINE):]
        reg_report(fr, chosen, y, f"+{fam} ({len(added)} feat)")

## Cell 12: All-families forward selection per slice (the headline run)

Pool every inversion-symmetry feature (buckling excluded on 124/200), seed with the baseline, and
forward-select up to 8 added features. This is the combined model - what broken-inversion-symmetry
descriptors do to the regression ceiling. Reported with the LOO R2 at each step so you can see where it
plateaus and pick the parsimonious size. The chosen feature lists per slice are saved.

In [ ]:
# Headline run. On the 99 slice we try THREE targets (max / mean / min alpha) because a rashba
# compound can have several alpha values across k-paths; only the 99 is multi-row, so 124/200 stay on
# alpha_max. mean is a smoother, different target - its R2 is NOT directly comparable to the max-based
# 0.66 headline; min is the most conservative. For the ~58 single-row rashba compounds all three are
# identical, so any difference is driven by the ~41 multi-row ones.
summary = {}
TARGETS = {"99": [("max", "alpha"), ("mean", "alpha_mean"), ("min", "alpha_min")],
           "124": [("max", "alpha")],
           "200": [("max", "alpha")]}

for sl in ["99", "124", "200"]:
    fr = slice_frame(sl).reset_index(drop=True)
    pool = [f for f in INVSYM_FEATS if not (sl != "99" and f in BUCKLING_99_ONLY)]
    for tname, tcol in TARGETS[sl]:
        y = fr[tcol].values
        print(f"\\n===== SLICE {sl}  target=alpha_{tname}  (n={len(fr)})  pool={len(pool)} =====")
        reg_report(fr, BASELINE, y, "baseline only")
        chosen = reg_forward(fr, y, pool, BASELINE, upto=len(BASELINE) + 8)
        best_r2 = -np.inf; best_k = len(BASELINE)
        for k in range(len(BASELINE), len(chosen) + 1):
            r2 = reg_report(fr, chosen[:k], y, f"baseline + {k - len(BASELINE)} invsym")
            if r2 > best_r2:
                best_r2, best_k = r2, k
        summary[f"{sl}_{tname}"] = dict(slice=sl, target=tname, best_r2=best_r2,
                                        n_feat=best_k, features=chosen[:best_k])
        print(f"  -> best LOO R2 = {best_r2:.3f} at {best_k - len(BASELINE)} added features")
        print(f"     features: {chosen[:best_k]}")

print("\\n================ SUMMARY ================")
for key, d in summary.items():
    print(f"  {key:10s}: best LOO R2 = {d['best_r2']:.3f}  ({d['n_feat'] - len(BASELINE)} invsym feats)")
pd.DataFrame([{k: v for k, v in d.items() if k != "features"} for d in summary.values()]).to_csv(
    os.path.join(RESULTS_DIR, "invsym_regression_summary.csv"), index=False)


## What to read from the output

- **Cell 9** tells you which inversion-symmetry descriptors actually correlate with alpha on the 99 and
  124, and which tolerance won - the empirical answer to 'does quantifying broken inversion help magnitude.'
- **Cell 11** isolates each family: inversion (A) and global (C) should carry the 99; mirror (B) is the
  one to watch on the 124 (the dresselhaus/non-polar compounds); structural (D) is buckling-driven on 99.
- **Cell 12** is the headline: baseline LOO R2 vs baseline+invsym, per slice. The comparison against the
  ~0.66 environment-only ceiling lives here.

Everything is LOO, forward-selected (small-sample safe), and buckling is excluded where it would be a
class constant. Paste Cells 9, 11, 12 back and we decide which descriptors graduate into the report.